# Actualización de Tablas Vida

## 1. Importación de paqueterías

In [21]:
import pandas as pd
import urllib
from sqlalchemy import create_engine, text


## 2. Conexión a SQL Server

In [22]:

# -----------------------------
# 1️⃣ Conexión a SQL Server Azure
# -----------------------------
params_azure = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=ikaldb.database.windows.net;"
    "DATABASE=actuarial;"
    "UID=CloudSA0052c2f7;"
    "PWD=ricostamales01#;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)
engine_azure = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params_azure}",
    fast_executemany=True
)
print("✅ Conexión SQL Server AZURE establecida.")

# -----------------------------
# 2️⃣ Conexión a SQL Server Local
# -----------------------------
params_local = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=IKAL14\\SQLEXPRESS;"
    "DATABASE=actuarial;"
    "Trusted_Connection=yes;"
)
engine_local = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params_local}",
    fast_executemany=True
)
print("✅ Conexión SQL Server Local establecida.")


✅ Conexión SQL Server AZURE establecida.
✅ Conexión SQL Server Local establecida.


## 3. Carga de datos

In [23]:
AñoMes = "202512"  # ← Cambia solo este valor cada mes
archivo_excel = rf"C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/{AñoMes[:4]}/{AñoMes}/{AñoMes}_Siniestros_Marine_PROCESADO.xlsx"
df_nuevo = pd.read_excel(archivo_excel)
print(f"✅ Datos leídos: '{archivo_excel}' ({len(df_nuevo):,} registros)")

✅ Datos leídos: 'C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/2025/202512/202512_Siniestros_Marine_PROCESADO.xlsx' (4,276 registros)


In [24]:
# Eliminar datos de la tabla 'transporte'
#print("🧹 Eliminando datos antiguos de la tabla 'transporte'...")
#with engine_azure.connect() as conn:
#    conn.execute(text("TRUNCATE TABLE dbo.transporte"))
#    conn.commit()
#with engine_local.connect() as conn:
#    conn.execute(text("TRUNCATE TABLE dbo.transporte"))
#    conn.commit()
#print("✅ Datos antiguos eliminados de la tabla 'transporte' en ambas bases de datos.")

# Eliminar datos de la tabla 'transportehist'
#print("🧹 Eliminando datos antiguos de la tabla 'transportehist'...")
#with engine_azure.connect() as conn:
#    conn.execute(text("TRUNCATE TABLE dbo.transportehist"))
#    conn.commit()
#with engine_local.connect() as conn:
#    conn.execute(text("TRUNCATE TABLE dbo.transportehist"))
#    conn.commit()
#print("✅ Datos antiguos eliminados de la tabla 'transportehist' en ambas bases de datos.")

## 4. Limpieza y carga de tabla transporte

In [25]:
# -----------------------------
# 3️⃣ Actualizar tabla 'transporte'
# -----------------------------
print("📊 Actualizando tabla 'transporte' (mes actual)...")
df_nuevo.to_sql('transporte', con=engine_azure, if_exists='replace', index=False, schema='dbo')
df_nuevo.to_sql('transporte', con=engine_local, if_exists='replace', index=False, schema='dbo')
print(f"✅ Tabla 'transporte' reemplazada con {len(df_nuevo):,} registros")


📊 Actualizando tabla 'transporte' (mes actual)...
✅ Tabla 'transporte' reemplazada con 4,276 registros


## 5. Inserción y actualización de tabla transportehist

In [26]:
# =============================================================================
# 5️⃣ INSERCIÓN EN TABLA TRANSPORTEHIST  (snapshot mensual + columna activo)
# =============================================================================
# Lógica:
#   - transporte      → se reemplaza cada mes con el mes actual
#   - transportehist  → se acumula: cada mes se insertan filas nuevas
#                       activo = 1  →  mes más reciente cargado
#                       activo = 0  →  meses anteriores
# =============================================================================

# 📋 PASO 1: Obtener estructura de tabla SQL
print("\n" + "="*80)
print("PASO 1: Obtener estructura de tabla SQL")
print("="*80)

query_columnas = """
SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'transportehist' AND TABLE_SCHEMA = 'dbo'
ORDER BY ORDINAL_POSITION
"""
df_cols_azure = pd.read_sql(query_columnas, con=engine_azure)
df_cols_local = pd.read_sql(query_columnas, con=engine_local)

columnas_sql_list = df_cols_azure['COLUMN_NAME'].tolist()
columnas_sql_upper = {c.upper(): c for c in columnas_sql_list}
columnas_fecha_sql = set(df_cols_azure[df_cols_azure['DATA_TYPE'].isin(
    ['date', 'datetime', 'datetime2'])]['COLUMN_NAME'].tolist())

def _get_len_map(df):
    m = (df['DATA_TYPE'].isin(['varchar', 'nvarchar', 'char', 'nchar']) &
         df['CHARACTER_MAXIMUM_LENGTH'].notna() & (df['CHARACTER_MAXIMUM_LENGTH'] > 0))
    return dict(zip(df.loc[m, 'COLUMN_NAME'], df.loc[m, 'CHARACTER_MAXIMUM_LENGTH'].astype(int)))

len_azure = _get_len_map(df_cols_azure)
len_local = _get_len_map(df_cols_local)
max_len_map = {col: min(v for v in [len_azure.get(col), len_local.get(col)] if v is not None)
               for col in set(len_azure) | set(len_local)}

# Agregar columna activo si no existe
for label, eng in [("Local", engine_local), ("Azure", engine_azure)]:
    with eng.begin() as conn:
        conn.execute(text("""
            IF COL_LENGTH('dbo.transportehist', 'activo') IS NULL
                ALTER TABLE dbo.transportehist ADD [activo] INT NOT NULL DEFAULT 1
        """))
print("✓ Columna [activo] verificada")

rename_map = {col: columnas_sql_upper[col.upper()]
              for col in df_nuevo.columns
              if col.upper() in columnas_sql_upper and col != columnas_sql_upper[col.upper()]}
df_nuevo_renamed = df_nuevo.rename(columns=rename_map)

columnas_sql    = set(columnas_sql_list)
columnas_excel  = set(df_nuevo_renamed.columns.tolist())
columnas_validas = sorted(list(columnas_excel & columnas_sql))
print(f"✓ Columnas válidas: {len(columnas_validas)}")

solo_excel = columnas_excel - columnas_sql
if solo_excel:
    print(f"⚠️  Solo en Excel (ignoradas): {sorted(list(solo_excel))}")

# 📋 PASO 2: Preparar datos del Excel
print("\n" + "="*80)
print("PASO 2: Preparar datos del Excel")
print("="*80)

df_transporte_limpio = df_nuevo_renamed[columnas_validas].copy()

cols_fecha_presentes = [c for c in columnas_fecha_sql if c in df_transporte_limpio.columns]
for col in cols_fecha_presentes:
    ser = df_transporte_limpio[col]
    if pd.api.types.is_datetime64_any_dtype(ser):
        continue
    if pd.api.types.is_integer_dtype(ser) or pd.api.types.is_float_dtype(ser):
        sample = ser.dropna()
        if len(sample) > 0 and not sample.between(1900, 2100).all():
            df_transporte_limpio[col] = pd.to_datetime(ser, unit='D', origin='1899-12-30', errors='coerce')
    else:
        df_transporte_limpio[col] = pd.to_datetime(ser, errors='coerce')

for col, max_len in max_len_map.items():
    if col in df_transporte_limpio.columns and df_transporte_limpio[col].dtype == object:
        if df_transporte_limpio[col].str.len().gt(max_len).any():
            df_transporte_limpio[col] = df_transporte_limpio[col].str[:max_len]

df_transporte_limpio['activo'] = 1
print(f"✓ {len(df_transporte_limpio):,} registros preparados con activo = 1")

# 📋 PASO 3: Cargar datos a tabla temporal
print("\n" + "="*80)
print("PASO 3: Cargar datos a tabla temporal")
print("="*80)

df_transporte_limpio.to_sql('temp_transportehist_mes', con=engine_local,
                             if_exists='replace', index=False, schema='dbo')
df_transporte_limpio.to_sql('temp_transportehist_mes', con=engine_azure,
                             if_exists='replace', index=False, schema='dbo')
print(f"✓ Tabla temporal creada con {len(df_transporte_limpio):,} registros")

# Corregir bigint→date en transportehist (columnas de año entero)
query_mismatch = """
SELECT t.COLUMN_NAME, t.DATA_TYPE AS tipo_hist, s.DATA_TYPE AS tipo_temp
FROM INFORMATION_SCHEMA.COLUMNS t
JOIN INFORMATION_SCHEMA.COLUMNS s
    ON t.COLUMN_NAME = s.COLUMN_NAME
   AND s.TABLE_NAME = 'temp_transportehist_mes' AND s.TABLE_SCHEMA = 'dbo'
WHERE t.TABLE_NAME = 'transportehist' AND t.TABLE_SCHEMA = 'dbo'
  AND t.DATA_TYPE <> s.DATA_TYPE
ORDER BY t.COLUMN_NAME
"""
mismatches = pd.read_sql(query_mismatch, con=engine_local)
cols_bigint_date = mismatches[
    (mismatches['tipo_hist'] == 'date') & (mismatches['tipo_temp'] == 'bigint')
]['COLUMN_NAME'].tolist()

if cols_bigint_date:
    print(f"Migrando columnas de año DATE→INT: {cols_bigint_date}")
    for col in cols_bigint_date:
        col_tmp = col.replace(" ", "_").replace("-", "_") + "_yr_tmp"
        for label, eng in [("Local", engine_local), ("Azure", engine_azure)]:
            with eng.begin() as conn:
                conn.execute(text(f"IF COL_LENGTH('dbo.transportehist','{col_tmp}') IS NOT NULL ALTER TABLE dbo.transportehist DROP COLUMN [{col_tmp}]"))
                conn.execute(text(f"ALTER TABLE dbo.transportehist ADD [{col_tmp}] INT NULL"))
                conn.execute(text(f"UPDATE dbo.transportehist SET [{col_tmp}] = YEAR([{col}])"))
                conn.execute(text(f"""
                    DECLARE @sql NVARCHAR(MAX) = ''
                    SELECT @sql = @sql + 'DROP INDEX [' + i.name + '] ON dbo.transportehist; '
                    FROM sys.indexes i
                    JOIN sys.index_columns ic ON i.object_id=ic.object_id AND i.index_id=ic.index_id
                    JOIN sys.columns c ON ic.object_id=c.object_id AND ic.column_id=c.column_id
                    WHERE OBJECT_NAME(i.object_id)='transportehist' AND c.name='{col}'
                      AND i.is_primary_key=0 AND i.is_unique_constraint=0
                    IF LEN(@sql) > 0 EXEC(@sql)
                """))
                conn.execute(text(f"ALTER TABLE dbo.transportehist DROP COLUMN [{col}]"))
                conn.execute(text(f"EXEC sp_rename @objname=N'dbo.transportehist.{col_tmp}', @newname=N'{col}', @objtype=N'COLUMN'"))
        print(f"  ✓ [{col}] → INT")

# 📋 PASO 4: Desactivar mes anterior e insertar mes nuevo
print("\n" + "="*80)
print("PASO 4: Actualizar activo e insertar registros del mes")
print("="*80)

mes_año = int(AñoMes[:4])
mes_num = int(AñoMes[4:6])

# Eliminar registros del mismo mes si ya existen (permite re-ejecución sin duplicados)
for label, eng in [("Local", engine_local), ("Azure", engine_azure)]:
    with eng.begin() as conn:
        deleted = conn.execute(text(f"""
            DELETE FROM dbo.transportehist
            WHERE YEAR([MES CARGA]) = {mes_año} AND MONTH([MES CARGA]) = {mes_num}
        """)).rowcount
    if deleted:
        print(f"🗑️  {deleted:,} registros de {AñoMes} eliminados en {label} (re-ejecución detectada)")

# Marcar todos los registros existentes como activo = 0
for label, eng in [("Local", engine_local), ("Azure", engine_azure)]:
    with eng.begin() as conn:
        deactivated = conn.execute(text(
            "UPDATE dbo.transportehist SET [activo] = 0 WHERE [activo] = 1"
        )).rowcount
    print(f"✓ {deactivated:,} registros anteriores → activo=0  ({label})")

# Insertar registros del mes nuevo con activo = 1
all_cols     = columnas_validas + ['activo']
cols_sql_str = ', '.join([f'[{c}]' for c in all_cols])
insert_sql   = f"""
INSERT INTO dbo.transportehist ({cols_sql_str})
SELECT {cols_sql_str} FROM dbo.temp_transportehist_mes
"""
for label, eng in [("Local", engine_local), ("Azure", engine_azure)]:
    with eng.begin() as conn:
        inserted = conn.execute(text(insert_sql)).rowcount
    print(f"✓ {inserted:,} registros nuevos insertados (activo=1)  ({label})")

# 📋 PASO 5: Resumen y limpieza
print("\n" + "="*80)
print("RESUMEN FINAL")
print("="*80)

resumen = pd.read_sql("""
    SELECT [activo], COUNT(*) AS registros
    FROM transportehist
    GROUP BY [activo]
    ORDER BY [activo] DESC
""", con=engine_azure)

print(f"transportehist Azure:")
for _, row in resumen.iterrows():
    estado = f"MES ACTIVO ({AñoMes})" if row['activo'] == 1 else "Histórico"
    print(f"   activo={int(row['activo'])}  →  {int(row['registros']):,} registros  ({estado})")

for eng, label in [(engine_local, "Local"), (engine_azure, "Azure")]:
    with eng.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS temp_transportehist_mes"))
    print(f"🧹 Tabla temporal eliminada en {label}")

print("="*80)



PASO 1: Obtener estructura de tabla SQL
✓ Columna [activo] verificada
✓ Columnas válidas: 38
⚠️  Solo en Excel (ignoradas): ['CONFIANZA', 'METODO_MATCH', 'PAIS', 'STATUS_ORIGINAL']

PASO 2: Preparar datos del Excel
✓ 4,276 registros preparados con activo = 1

PASO 3: Cargar datos a tabla temporal
✓ Tabla temporal creada con 4,276 registros

PASO 4: Actualizar activo e insertar registros del mes
✓ 4,256 registros anteriores → activo=0  (Local)
✓ 4,256 registros anteriores → activo=0  (Azure)
✓ 4,276 registros nuevos insertados (activo=1)  (Local)
✓ 4,276 registros nuevos insertados (activo=1)  (Azure)

RESUMEN FINAL
transportehist Azure:
   activo=1  →  4,276 registros  (MES ACTIVO (202512))
   activo=0  →  8,507 registros  (Histórico)
🧹 Tabla temporal eliminada en Local
🧹 Tabla temporal eliminada en Azure
